# Thompson Sampling for the Multi-Armed Bandit Problem

**Author:** François Cinotti  

---

This notebook introduces **Thompson Sampling**, a classic algorithm for solving the *exploration-exploitation dilemma* in sequential decision-making. We will:

1. Set up a three-armed bandit task
2. Implement vanilla Thompson Sampling and show it works in a stationary environment
3. Show why it fails when reward probabilities change over time
4. Introduce two extensions — **Dynamic Thompson Sampling (DTS)** and **Sliding Window Thompson Sampling (SWTS)** — that handle non-stationary environments

This notebook accompanies the paper:  
> Cinotti et al. (2024). *Regulation of reinforcement learning parameters captures long-term changes in rat behaviour.* European Journal of Neuroscience.

## Imports

We use standard Python scientific libraries. Make sure you have them installed (`pip install numpy matplotlib scipy`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

# Set a random seed for reproducibility
np.random.seed(42)

# Use a clean plot style
plt.style.use('seaborn-v0_8-whitegrid')

: 

---
## Part 1: The Three-Armed Bandit Task

### What is a bandit task?

Imagine you are in a casino facing three slot machines ("arms"). Each machine pays out a reward with some unknown probability. Your goal is to maximise your total reward over a series of trials.

This is the **exploration-exploitation dilemma**:
- **Exploit**: keep pulling the arm you currently believe is best
- **Explore**: try other arms in case one of them is actually better

A purely greedy strategy (always exploit) risks getting stuck on a suboptimal arm. A purely random strategy (always explore) wastes trials on bad arms. Good algorithms balance the two.

### Our task

We have **three arms** with the following reward probabilities:

| Arm | Reward probability |
|-----|-------------------|
| 0   | 0.6 ✓ (best)      |
| 1   | 0.2               |
| 2   | 0.2               |

On each trial, pulling arm $i$ yields a reward of 1 with probability $p_i$, and 0 otherwise. The agent does not know these probabilities — it must learn them from experience.

In [ ]:
# Define the true reward probabilities for each arm
# The agent does NOT have access to these — they are the ground truth we use to simulate the task
TRUE_PROBS = [0.6, 0.2, 0.2]
N_ARMS = len(TRUE_PROBS)

def pull_arm(arm, probs):
    """
    Simulate pulling an arm.
    Returns 1 (reward) with probability probs[arm], else 0.
    """
    return int(np.random.rand() < probs[arm])

---
## Part 2: Thompson Sampling

### The key idea

Instead of storing a single estimate of each arm's reward probability, Thompson Sampling maintains a **probability distribution** over what the true reward probability might be. This distribution reflects both our current best guess and our uncertainty about it.

For binary (0/1) rewards, the natural choice is the **Beta distribution**, $\text{Beta}(\alpha, \beta)$, where:
- $\alpha$ = number of successes (rewards) observed + 1
- $\beta$ = number of failures (no reward) observed + 1

The Beta distribution is defined on $[0, 1]$, making it a natural way to represent uncertainty about a probability. Initially, with no observations, we set $\alpha = \beta = 1$, which gives a **uniform distribution** — meaning we consider all reward probabilities equally likely.

### The algorithm

On each trial:

1. **Sample** one value from the Beta distribution of each arm
2. **Select** the arm with the highest sampled value
3. **Observe** the reward (0 or 1)
4. **Update** the Beta distribution of the chosen arm:
   - If reward = 1: $\alpha_i \leftarrow \alpha_i + 1$
   - If reward = 0: $\beta_i \leftarrow \beta_i + 1$

Arms that have been sampled little will have **wide, flat distributions**, making them more likely to produce a high random sample — this naturally drives exploration. Arms with many observations will have **narrow distributions**, reflecting more certainty about their value.

In [ ]:
def thompson_sampling(n_trials, probs):
    """
    Run vanilla Thompson Sampling on a bandit task.
    
    Parameters
    ----------
    n_trials : int
        Number of trials to run
    probs : list of float
        True reward probability of each arm
    
    Returns
    -------
    choices : array of int
        Which arm was chosen on each trial
    rewards : array of int
        Reward received on each trial
    alphas : array of shape (n_trials, n_arms)
        Evolution of alpha parameters over trials
    betas : array of shape (n_trials, n_arms)
        Evolution of beta parameters over trials
    """
    n_arms = len(probs)
    
    # Initialise Beta distribution parameters
    # alpha[i] = successes + 1, beta_params[i] = failures + 1
    # Starting at (1, 1) gives a uniform prior: we have no preference for any arm
    alpha = np.ones(n_arms)
    beta_params = np.ones(n_arms)
    
    # Storage for results
    choices = np.zeros(n_trials, dtype=int)
    rewards = np.zeros(n_trials, dtype=int)
    alphas = np.zeros((n_trials, n_arms))
    betas = np.zeros((n_trials, n_arms))
    
    for t in range(n_trials):
        
        # Step 1: Sample one value from each arm's Beta distribution
        # Arms we know little about have wide distributions and are more likely
        # to produce a high sample, encouraging exploration
        samples = [np.random.beta(alpha[i], beta_params[i]) for i in range(n_arms)]
        
        # Step 2: Choose the arm with the highest sample
        choice = np.argmax(samples)
        
        # Step 3: Pull the arm and observe the reward
        reward = pull_arm(choice, probs)
        
        # Step 4: Update the Beta distribution of the chosen arm
        if reward == 1:
            alpha[choice] += 1   # One more success
        else:
            beta_params[choice] += 1  # One more failure
        
        # Store results for this trial
        choices[t] = choice
        rewards[t] = reward
        alphas[t] = alpha.copy()
        betas[t] = beta_params.copy()
    
    return choices, rewards, alphas, betas

### Visualising the Beta distributions

Before running the full algorithm, let's build some intuition by visualising how the Beta distribution changes as we accumulate observations.

In [ ]:
# Plot Beta distributions for different numbers of observations
x = np.linspace(0, 1, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Beta distributions at different stages of learning', fontsize=13)

# Scenario 1: No observations yet — uniform prior
axes[0].plot(x, beta_dist.pdf(x, 1, 1), 'k', lw=2)
axes[0].set_title('No observations\nBeta(1, 1)')
axes[0].set_xlabel('Reward probability')
axes[0].set_ylabel('Density')

# Scenario 2: A few observations — broad distribution, still uncertain
axes[1].plot(x, beta_dist.pdf(x, 4, 3), 'steelblue', lw=2, label='3 rewards, 2 failures')
axes[1].plot(x, beta_dist.pdf(x, 2, 5), 'tomato', lw=2, label='1 reward, 4 failures')
axes[1].set_title('Few observations\n(still uncertain)')
axes[1].set_xlabel('Reward probability')
axes[1].legend(fontsize=8)

# Scenario 3: Many observations — narrow distribution, confident estimate
axes[2].plot(x, beta_dist.pdf(x, 37, 25), 'steelblue', lw=2, label='36 rewards, 24 failures')
axes[2].plot(x, beta_dist.pdf(x, 11, 51), 'tomato', lw=2, label='10 rewards, 50 failures')
axes[2].set_title('Many observations\n(confident estimate)')
axes[2].set_xlabel('Reward probability')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

### Running Thompson Sampling on our task

Now let's run the full algorithm and see how well it identifies the best arm.

In [ ]:
# Run Thompson Sampling for 500 trials
N_TRIALS = 500
choices, rewards, alphas, betas = thompson_sampling(N_TRIALS, TRUE_PROBS)

# Compute the proportion of trials on which the best arm (arm 0) was chosen
# We use a rolling window of 20 trials to smooth the curve
window = 20
best_arm_chosen = (choices == 0).astype(float)
smoothed = np.convolve(best_arm_chosen, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left plot: proportion of trials choosing the best arm over time
axes[0].plot(smoothed, color='steelblue', lw=2)
axes[0].axhline(0.33, color='gray', linestyle='--', label='Chance (1/3)')
axes[0].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Proportion choosing best arm')
axes[0].set_title('Learning curve: does the agent find the best arm?')
axes[0].legend()
axes[0].set_ylim(0, 1.05)

# Right plot: evolution of the Beta distributions at the end of the experiment
x = np.linspace(0, 1, 200)
colors = ['steelblue', 'tomato', 'green']
for i in range(N_ARMS):
    a, b = alphas[-1, i], betas[-1, i]
    axes[1].plot(x, beta_dist.pdf(x, a, b), color=colors[i], lw=2,
                 label=f'Arm {i} (true p={TRUE_PROBS[i]})')
    axes[1].axvline(TRUE_PROBS[i], color=colors[i], linestyle='--', alpha=0.5)
axes[1].set_xlabel('Reward probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Beta distributions after 500 trials')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Final arm selection counts: Arm 0: {np.sum(choices==0)}, "
      f"Arm 1: {np.sum(choices==1)}, Arm 2: {np.sum(choices==2)}")

As expected, Thompson Sampling quickly learns to favour arm 0, and the Beta distributions converge around the true reward probabilities.

---
## Part 3: The Twist — Non-Stationary Reward Probabilities

### The problem

In the real world — and in many neuroscience experiments — reward probabilities **change over time**. In the rat experiment from Cinotti et al. (2024), the identity of the best lever changed every 24 trials without warning.

The key issue with vanilla Thompson Sampling is that it accumulates evidence **indefinitely**. After many trials, the Beta distributions become very narrow and confident. When the reward probabilities then switch, the agent is slow to notice because it takes many new observations to meaningfully shift a distribution that is already very concentrated.

### Our non-stationary task

We will now simulate a task where the reward probabilities **rotate** every 100 trials:

| Block | Arm 0 | Arm 1 | Arm 2 |
|-------|-------|-------|-------|
| 1     | **0.6** | 0.2 | 0.2 |
| 2     | 0.2 | **0.6** | 0.2 |
| 3     | 0.2 | 0.2 | **0.6** |
| 4     | **0.6** | 0.2 | 0.2 |
| ...   | ... | ... | ... |

In [ ]:
def make_nonstationary_schedule(n_trials, block_size=100):
    """
    Generate a schedule of reward probabilities that rotate every block_size trials.
    The best arm rotates: 0 -> 1 -> 2 -> 0 -> ...
    
    Returns
    -------
    prob_schedule : array of shape (n_trials, n_arms)
        True reward probabilities at each trial
    best_arm_schedule : array of int
        Which arm is best at each trial
    """
    n_arms = 3
    prob_schedule = np.zeros((n_trials, n_arms))
    best_arm_schedule = np.zeros(n_trials, dtype=int)
    
    for t in range(n_trials):
        # Which block are we in?
        block = (t // block_size) % n_arms
        
        # The best arm rotates with each block
        probs = [0.2, 0.2, 0.2]
        probs[block] = 0.6
        
        prob_schedule[t] = probs
        best_arm_schedule[t] = block
    
    return prob_schedule, best_arm_schedule

# Generate the schedule
N_TRIALS_NS = 600  # 6 blocks of 100 trials
BLOCK_SIZE = 100
prob_schedule, best_arm_schedule = make_nonstationary_schedule(N_TRIALS_NS, BLOCK_SIZE)

print("Reward probabilities in each block:")
for b in range(6):
    t = b * BLOCK_SIZE
    print(f"  Block {b+1} (trials {t}-{t+BLOCK_SIZE-1}): {prob_schedule[t].tolist()}  "
          f"→ best arm: {best_arm_schedule[t]}")

In [ ]:
def thompson_sampling_nonstationary(n_trials, prob_schedule):
    """
    Run vanilla Thompson Sampling on a non-stationary bandit task.
    The reward probabilities change at each trial according to prob_schedule.
    
    Note: the agent does NOT know when or how probabilities change.
    """
    n_arms = prob_schedule.shape[1]
    
    # Initialise Beta distributions with uniform priors
    alpha = np.ones(n_arms)
    beta_params = np.ones(n_arms)
    
    choices = np.zeros(n_trials, dtype=int)
    rewards = np.zeros(n_trials, dtype=int)
    
    for t in range(n_trials):
        
        # Sample from each arm's Beta distribution
        samples = [np.random.beta(alpha[i], beta_params[i]) for i in range(n_arms)]
        
        # Choose the arm with the highest sample
        choice = np.argmax(samples)
        
        # Pull arm using CURRENT reward probabilities
        reward = pull_arm(choice, prob_schedule[t])
        
        # Update Beta distribution — accumulating ALL past evidence
        # This is the key weakness: old evidence from previous blocks is never forgotten
        if reward == 1:
            alpha[choice] += 1
        else:
            beta_params[choice] += 1
        
        choices[t] = choice
        rewards[t] = reward
    
    return choices, rewards

# Run vanilla Thompson Sampling on the non-stationary task
choices_vanilla, rewards_vanilla = thompson_sampling_nonstationary(N_TRIALS_NS, prob_schedule)

In [ ]:
def compute_performance(choices, best_arm_schedule, window=20):
    """
    Compute smoothed proportion of trials on which the correct arm was chosen.
    """
    correct = (choices == best_arm_schedule).astype(float)
    smoothed = np.convolve(correct, np.ones(window)/window, mode='valid')
    return smoothed

perf_vanilla = compute_performance(choices_vanilla, best_arm_schedule)

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(perf_vanilla, color='steelblue', lw=2, label='Vanilla Thompson Sampling')
ax.axhline(0.33, color='gray', linestyle='--', label='Chance (1/3)')

# Mark block boundaries
for b in range(1, N_TRIALS_NS // BLOCK_SIZE):
    ax.axvline(b * BLOCK_SIZE, color='red', linestyle=':', alpha=0.7)
    ax.text(b * BLOCK_SIZE + 2, 0.05, f'Block {b+1}', color='red', fontsize=8)

ax.set_xlabel('Trial')
ax.set_ylabel('Proportion choosing correct arm')
ax.set_title('Vanilla Thompson Sampling on a non-stationary task\n'
             'Red dotted lines = block changes (reward probabilities switch)')
ax.legend()
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

### What do we observe?

Vanilla Thompson Sampling performs well in the **first block** — it successfully identifies the best arm. But after the block change, performance **drops and recovers slowly**, or may not recover at all. 

The reason: by the end of the first block, arm 0 has accumulated many successes. Its Beta distribution is narrow and centred near 0.6. Even after the block switches and arm 0 starts returning fewer rewards, it takes many trials of failure to shift the now-confident distribution. The algorithm is **stuck in the past**.

This motivates two extensions designed specifically for non-stationary environments.

---
## Part 4: Extensions for Non-Stationary Environments

### 4.1 Dynamic Thompson Sampling (DTS)

Proposed by Gupta et al. (2011), DTS introduces a **minimum variance constraint** $C$ on the Beta distributions. At each trial, after updating the distributions, any distribution that has become too narrow (i.e. its variance has dropped below $C$) is **widened** by rescaling its parameters while preserving the mean.

The variance of a $\text{Beta}(\alpha, \beta)$ distribution is:

$$\text{Var} = \frac{\alpha \beta}{(\alpha + \beta)^2 (\alpha + \beta + 1)}$$

If this variance falls below $C$, the parameters $\alpha$ and $\beta$ are rescaled by a factor $s < 1$ such that the variance is restored to $C$. This prevents the distributions from becoming too confident, ensuring the agent remains somewhat open to revising its beliefs.

In [ ]:
def beta_variance(a, b):
    """Compute the variance of a Beta(a, b) distribution."""
    return (a * b) / ((a + b)**2 * (a + b + 1))

def rescale_to_variance(a, b, target_var):
    """
    Rescale Beta(a, b) parameters to achieve target_var variance,
    while preserving the mean a/(a+b).
    
    We scale both parameters by a factor s: (sa, sb) has the same mean
    but different variance. We solve for s numerically.
    """
    mean = a / (a + b)
    # Binary search for the right scaling factor s
    s_low, s_high = 1e-6, 1.0
    for _ in range(50):  # 50 iterations is more than enough
        s_mid = (s_low + s_high) / 2
        var_mid = beta_variance(s_mid * a, s_mid * b)
        if var_mid < target_var:
            s_high = s_mid
        else:
            s_low = s_mid
    s = (s_low + s_high) / 2
    return s * a, s * b

def dynamic_thompson_sampling(n_trials, prob_schedule, C):
    """
    Dynamic Thompson Sampling (Gupta et al., 2011).
    
    Parameters
    ----------
    n_trials : int
    prob_schedule : array of shape (n_trials, n_arms)
    C : float
        Minimum variance threshold. Higher C = more forgetting = faster adaptation.
    """
    n_arms = prob_schedule.shape[1]
    alpha = np.ones(n_arms)
    beta_params = np.ones(n_arms)
    
    choices = np.zeros(n_trials, dtype=int)
    rewards = np.zeros(n_trials, dtype=int)
    
    for t in range(n_trials):
        
        # Step 1: Sample from each arm's Beta distribution
        samples = [np.random.beta(alpha[i], beta_params[i]) for i in range(n_arms)]
        choice = np.argmax(samples)
        
        # Step 2: Pull the arm
        reward = pull_arm(choice, prob_schedule[t])
        
        # Step 3: Update the chosen arm's distribution
        if reward == 1:
            alpha[choice] += 1
        else:
            beta_params[choice] += 1
        
        # Step 4: DTS-specific — enforce minimum variance on ALL arms
        # If any arm's distribution has become too narrow (overconfident),
        # rescale it to maintain a minimum variance of C
        for i in range(n_arms):
            if beta_variance(alpha[i], beta_params[i]) < C:
                alpha[i], beta_params[i] = rescale_to_variance(alpha[i], beta_params[i], C)
        
        choices[t] = choice
        rewards[t] = reward
    
    return choices, rewards

### 4.2 Sliding Window Thompson Sampling (SWTS)

Proposed by Trovò et al. (2020), SWTS takes a simpler approach: instead of using all past observations, it only counts observations from the **last $T$ trials**. Older observations are simply discarded.

This means:
- $\alpha_i$ = number of successes for arm $i$ in the last $T$ trials + 1
- $\beta_i$ = number of failures for arm $i$ in the last $T$ trials + 1

The parameter $T$ controls the trade-off: a small $T$ adapts quickly but is noisy; a large $T$ is more stable but slower to adapt.

In [ ]:
def sliding_window_thompson_sampling(n_trials, prob_schedule, T):
    """
    Sliding Window Thompson Sampling (Trovò et al., 2020).
    
    Parameters
    ----------
    n_trials : int
    prob_schedule : array of shape (n_trials, n_arms)
    T : int
        Window size. Only the last T trials are used to update beliefs.
    """
    n_arms = prob_schedule.shape[1]
    
    # Store the history of choices and rewards for the sliding window
    choice_history = []
    reward_history = []
    
    choices = np.zeros(n_trials, dtype=int)
    rewards = np.zeros(n_trials, dtype=int)
    
    for t in range(n_trials):
        
        # Step 1: Compute Beta parameters from the sliding window only
        # Look back at most T trials
        window_start = max(0, t - T)
        
        alpha = np.ones(n_arms)  # Start from 1 (uniform prior)
        beta_params = np.ones(n_arms)
        
        # Count successes and failures within the window for each arm
        for past_t in range(window_start, t):
            past_choice = choice_history[past_t - window_start + max(0, window_start - (t - T))]
            past_reward = reward_history[past_t - window_start + max(0, window_start - (t - T))]
            if past_reward == 1:
                alpha[past_choice] += 1
            else:
                beta_params[past_choice] += 1
        
        # Step 2: Sample and choose
        samples = [np.random.beta(alpha[i], beta_params[i]) for i in range(n_arms)]
        choice = np.argmax(samples)
        
        # Step 3: Pull arm
        reward = pull_arm(choice, prob_schedule[t])
        
        # Step 4: Store in history (keep only the last T entries)
        choice_history.append(choice)
        reward_history.append(reward)
        if len(choice_history) > T:
            choice_history.pop(0)
            reward_history.pop(0)
        
        choices[t] = choice
        rewards[t] = reward
    
    return choices, rewards

### Comparing all three algorithms

In [ ]:
# Run DTS and SWTS with example parameter values
# C = minimum variance for DTS (higher = more forgetting)
# T = window size for SWTS (smaller = faster forgetting)
C_VALUE = 0.01
T_VALUE = 50

choices_dts, rewards_dts = dynamic_thompson_sampling(N_TRIALS_NS, prob_schedule, C=C_VALUE)
choices_swts, rewards_swts = sliding_window_thompson_sampling(N_TRIALS_NS, prob_schedule, T=T_VALUE)

# Compute smoothed performance for all three algorithms
perf_vanilla = compute_performance(choices_vanilla, best_arm_schedule)
perf_dts = compute_performance(choices_dts, best_arm_schedule)
perf_swts = compute_performance(choices_swts, best_arm_schedule)

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(perf_vanilla, color='steelblue', lw=2, label='Vanilla TS')
ax.plot(perf_dts, color='tomato', lw=2, label=f'DTS (C={C_VALUE})')
ax.plot(perf_swts, color='green', lw=2, label=f'SWTS (T={T_VALUE})')
ax.axhline(0.33, color='gray', linestyle='--', alpha=0.7, label='Chance (1/3)')

# Mark block boundaries
for b in range(1, N_TRIALS_NS // BLOCK_SIZE):
    ax.axvline(b * BLOCK_SIZE, color='black', linestyle=':', alpha=0.4)

ax.set_xlabel('Trial')
ax.set_ylabel('Proportion choosing correct arm')
ax.set_title('Vanilla TS vs Dynamic TS vs Sliding Window TS\n'
             'Dotted vertical lines = block changes')
ax.legend()
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

### Exploring the effect of parameters

The behaviour of DTS and SWTS depends heavily on their parameters. Let's explore how different values affect performance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DTS: vary C
C_values = [0.001, 0.01, 0.05]
colors_c = ['#d62728', '#ff7f0e', '#bcbd22']
for C, col in zip(C_values, colors_c):
    ch, _ = dynamic_thompson_sampling(N_TRIALS_NS, prob_schedule, C=C)
    perf = compute_performance(ch, best_arm_schedule)
    axes[0].plot(perf, color=col, lw=2, label=f'C={C}')

axes[0].axhline(0.33, color='gray', linestyle='--', alpha=0.7, label='Chance')
for b in range(1, N_TRIALS_NS // BLOCK_SIZE):
    axes[0].axvline(b * BLOCK_SIZE, color='black', linestyle=':', alpha=0.3)
axes[0].set_title('Dynamic TS: effect of minimum variance C\n(higher C = more forgetting)')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Proportion correct')
axes[0].legend()
axes[0].set_ylim(0, 1.05)

# SWTS: vary T
T_values = [20, 50, 150]
colors_t = ['#1f77b4', '#17becf', '#aec7e8']
for T, col in zip(T_values, colors_t):
    ch, _ = sliding_window_thompson_sampling(N_TRIALS_NS, prob_schedule, T=T)
    perf = compute_performance(ch, best_arm_schedule)
    axes[1].plot(perf, color=col, lw=2, label=f'T={T}')

axes[1].axhline(0.33, color='gray', linestyle='--', alpha=0.7, label='Chance')
for b in range(1, N_TRIALS_NS // BLOCK_SIZE):
    axes[1].axvline(b * BLOCK_SIZE, color='black', linestyle=':', alpha=0.3)
axes[1].set_title('Sliding Window TS: effect of window size T\n(smaller T = faster forgetting)')
axes[1].set_xlabel('Trial')
axes[1].set_ylabel('Proportion correct')
axes[1].legend()
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## Summary

In this notebook we have:

1. **Introduced the three-armed bandit problem** as a model of the exploration-exploitation dilemma
2. **Implemented vanilla Thompson Sampling**, which represents uncertainty about arm values using Beta distributions and naturally balances exploration and exploitation
3. **Demonstrated its failure** in a non-stationary environment: because it accumulates evidence indefinitely, it cannot adapt when reward probabilities change
4. **Introduced two extensions**:
   - **Dynamic Thompson Sampling (DTS)**: enforces a minimum variance $C$ on all distributions, preventing overconfidence
   - **Sliding Window Thompson Sampling (SWTS)**: only uses the last $T$ trials, discarding old evidence

Both extensions adapt faster to environmental changes than vanilla TS, at the cost of being noisier when the environment is stable. The choice of $C$ or $T$ represents a fundamental trade-off between **stability** and **plasticity**.

Interestingly, as discussed in Cinotti et al. (2024), even these extended Thompson Sampling algorithms fail to capture rat behaviour in a similar task: they tend to learn either too fast or to converge to performance levels far above what the animals achieved — motivating the use of Q-learning based meta-learning models instead.

---
## References

- Cinotti, F., Coutureau, E., Khamassi, M., Marchand, A. R., & Girard, B. (2024). Regulation of reinforcement learning parameters captures long-term changes in rat behaviour. *European Journal of Neuroscience*.
- Gupta, N., Granmo, O.-C., & Agrawala, A. (2011). Thompson sampling for dynamic multi-armed bandits. *ICMLA*.
- Trovò, F., Paladino, S., Restelli, M., & Gatti, N. (2020). Sliding-window Thompson sampling for non-stationary settings. *Journal of Artificial Intelligence Research*, 68, 311–364.